# Análise exploratória dengue

Notebook separado para os gráficos e análises feitos a partir do dataframe tratado.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks_limpeza", Path.cwd().parent / "notebooks_limpeza"):
    if (candidate / "dengue_data_cleaner.py").exists():
        sys.path.insert(0, str(candidate))
        break

from dengue_data_cleaner import DengueDataCleaner


ANALISE_DIR = Path.cwd() if Path.cwd().name == "analise_dados" else Path.cwd() / "analise_dados"
IMAGES_DIR = ANALISE_DIR / "imagens"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
cleaner = DengueDataCleaner()
df_tratado = cleaner.transformar()
df = df_tratado.copy()

print("Tratado:", df_tratado.shape)


In [ ]:
df_tratado["final_classification"].value_counts()


In [ ]:
import matplotlib.pyplot as plt
import textwrap

df_confirmados = df[df["final_classification"] == 1].copy()

base_ocupacao = df_confirmados[["occupation_name", "sex_label"]].dropna().copy()
base_ocupacao["occupation_name"] = base_ocupacao["occupation_name"].astype(str).str.strip()
base_ocupacao["sex_label"] = base_ocupacao["sex_label"].astype(str).str.strip()

base_ocupacao = base_ocupacao[base_ocupacao["occupation_name"].str.lower() != "ignorado"]
base_ocupacao = base_ocupacao[base_ocupacao["sex_label"].isin(["Feminino", "Masculino"])]

def top_ocupacoes_por_sexo(sexo):
    return (
        base_ocupacao[base_ocupacao["sex_label"] == sexo]["occupation_name"]
        .value_counts()
        .head(10)
        .sort_values()
    )

nomes_ocupacoes_curto = {
    "APOSENTADO/PENSIONISTA": "Pensionista",
    "DESEMPREGADO CRONICO OU CUJA OCUPACAO HABITUAL NAO FOI": "Desempregado crônico",
    "EMPREGADO DOMESTICO NOS SERVICOS GERAIS": "Empregado doméstico",
    "EMPREGADO DOMESTICO DIARISTA": "Diarista",
    "TRABALHADOR AGROPECUARIO EM GERAL": "Trabalhador agropecuário",
    "PROFESSOR DA EDUCACAO DE JOVENS E ADULTOS DO ENSINO": "Professor EJA",
    "VENDEDOR DE COMERCIO VAREJISTA": "Vendedor varejista",
    "REPRESENTANTE COMERCIAL AUTONOMO": "Representante comercial",
    "MOTORISTA DE CAMINHAO (ROTAS REGIONAIS E INTERNACIONAIS)": "Motorista de caminhão",
    "TRABALHADOR VOLANTE DA AGRICULTURA": "Trabalhador volante agrícola",
}

def nome_ocupacao_curto(nome):
    return nomes_ocupacoes_curto.get(nome, nome.title())

def quebrar_labels(labels, largura=32):
    return ["\n".join(textwrap.wrap(nome_ocupacao_curto(label), width=largura)) for label in labels]

def plot_ocupacoes_por_sexo(dados, titulo, cor, arquivo):
    fig, ax = plt.subplots(figsize=(10, 6.2))
    labels = quebrar_labels(dados.index)
    bars = ax.barh(labels, dados.values, color=cor, alpha=0.9)

    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Número de casos confirmados", fontsize=9)
    ax.set_ylabel("")
    ax.grid(axis="x", linestyle="--", alpha=0.22)
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(axis="x", length=0, labelsize=8)
    ax.tick_params(axis="y", length=0, labelsize=8)

    maior_valor = dados.max()
    for bar in bars:
        largura = bar.get_width()
        ax.text(
            largura + maior_valor * 0.012,
            bar.get_y() + bar.get_height() / 2,
            f"{int(largura):,}".replace(",", "."),
            va="center",
            ha="left",
            fontsize=8,
            color="#334155",
        )

    ax.set_xlim(0, maior_valor * 1.18)
    plt.tight_layout()
    fig.savefig(IMAGES_DIR / arquivo, dpi=200, bbox_inches="tight")
    plt.show()

ocupacoes_feminino = top_ocupacoes_por_sexo("Feminino")
ocupacoes_masculino = top_ocupacoes_por_sexo("Masculino")

plot_ocupacoes_por_sexo(ocupacoes_feminino, "Ocupações com Maior Registro de Casos Confirmados - Feminino", "#0f766e", "ocupacoes_confirmadas_feminino.png")
plot_ocupacoes_por_sexo(ocupacoes_masculino, "Ocupações com Maior Registro de Casos Confirmados - Masculino", "#2563eb", "ocupacoes_confirmadas_masculino.png")


In [ ]:
import matplotlib.pyplot as plt

df_confirmados = df[df["final_classification"] == 1]

casos = (
    df_confirmados["sex_label"]
    .dropna()
    .astype(str)
    .str.strip()
)

casos = casos[casos.str.lower() != "ignorado"]
casos = casos.value_counts().reindex(["Feminino", "Masculino"]).dropna()
percentuais = (casos / casos.sum()) * 100

# Estilo Diacronic: simples, limpo e com leitura direta dos valores.
fig, ax = plt.subplots(figsize=(9, 5))
cores = ["#0f766e", "#2563eb"]
posicoes = [0.42, 0.0]
bars = ax.barh(posicoes, casos.values, color=cores, alpha=0.9, height=0.24)
ax.set_yticks(posicoes)
ax.set_yticklabels(casos.index)
ax.set_ylim(-0.22, 0.64)

ax.set_title("Distribuição dos Casos Confirmados por Sexo", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0)
ax.tick_params(axis="y", length=0, labelsize=11)

limite_direita = casos.max() * 1.18
ax.set_xlim(0, limite_direita)

for bar, percentual in zip(bars, percentuais):
    largura = bar.get_width()
    label = f"{int(largura):,}".replace(",", ".")
    ax.text(
        largura + casos.max() * 0.025,
        bar.get_y() + bar.get_height() / 2,
        f"{label} ({percentual:.1f}%)",
        va="center",
        ha="left",
        fontsize=10,
        color="#334155",
    )

plt.tight_layout()
fig.savefig(IMAGES_DIR / "casos_confirmados_por_sexo.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# População por estado - IBGE, estimativa de 2025.
pop_estado = {
    "Rondônia": 1751950,
    "Acre": 884372,
    "Amazonas": 4321616,
    "Roraima": 738772,
    "Pará": 8711196,
    "Amapá": 806517,
    "Tocantins": 1586859,
    "Maranhão": 7018211,
    "Piauí": 3384547,
    "Ceará": 9268836,
    "Rio Grande do Norte": 3455236,
    "Paraíba": 4164468,
    "Pernambuco": 9562007,
    "Alagoas": 3220848,
    "Sergipe": 2299425,
    "Bahia": 14870907,
    "Minas Gerais": 21393441,
    "Espírito Santo": 4126854,
    "Rio de Janeiro": 17223547,
    "São Paulo": 46081801,
    "Paraná": 11890517,
    "Santa Catarina": 8187029,
    "Rio Grande do Sul": 11233263,
    "Mato Grosso do Sul": 2924631,
    "Mato Grosso": 3893659,
    "Goiás": 7423629,
    "Distrito Federal": 2996899,
}

df_confirmados = df[df["final_classification"] == 1]

casos_estado = (
    df_confirmados["residence_state_label"]
    .value_counts(dropna=False)
    .rename_axis("estado")
    .reset_index(name="casos")
)

base = casos_estado.copy()
base["populacao"] = base["estado"].map(pop_estado)
base = base.dropna(subset=["populacao"])
base = base[base["populacao"] > 0]
base["casos_por_100k"] = (base["casos"] / base["populacao"]) * 100000

top_10_100k = base.sort_values("casos_por_100k", ascending=False).head(10).sort_values("casos_por_100k")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_10_100k["estado"], top_10_100k["casos_por_100k"], color="#0f766e", alpha=0.9)

ax.set_title("Estados com Maior Taxa de Casos Confirmados", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Casos confirmados por 100 mil habitantes", fontsize=9)
ax.set_ylabel("")
ax.grid(axis="x", linestyle="--", alpha=0.22)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0, labelsize=8)
ax.tick_params(axis="y", length=0, labelsize=9)

maior_valor = top_10_100k["casos_por_100k"].max()
for bar in bars:
    largura = bar.get_width()
    ax.text(
        largura + maior_valor * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{largura:.1f}",
        va="center",
        ha="left",
        fontsize=8,
        color="#334155",
    )

ax.set_xlim(0, maior_valor * 1.16)
plt.tight_layout()
fig.savefig(IMAGES_DIR / "casos_confirmados_por_100_mil_habitantes.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df["notification_date"] = pd.to_datetime(df["notification_date"], errors="coerce")
df["mes"] = df["notification_date"].dt.month

ordem_meses = list(range(1, 13))
nomes_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

casos_mes = (
    df["mes"]
    .value_counts()
    .reindex(ordem_meses)
    .fillna(0)
)

cores_estacoes = {
    "Verão": "#facc15",
    "Outono": "#f97316",
    "Inverno": "#7dd3fc",
    "Primavera": "#f472b6",
}

estacoes_mes = [
    "Verão", "Verão",
    "Outono", "Outono", "Outono",
    "Inverno", "Inverno", "Inverno",
    "Primavera", "Primavera", "Primavera",
    "Verão",
]
cores = [cores_estacoes[estacao] for estacao in estacoes_mes]

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(nomes_meses, casos_mes.values, color=cores, alpha=0.9, width=0.62)

ax.set_title("Total de Casos de Dengue por Mês entre 2017 a 2019", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.22)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0, labelsize=9)
ax.tick_params(axis="y", length=0, labelsize=8)

for bar in bars:
    altura = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        altura,
        f"{int(altura / 1000)} mil",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#334155",
    )

legenda = [
    plt.Rectangle((0, 0), 1, 1, color=cor, alpha=0.9, label=estacao)
    for estacao, cor in cores_estacoes.items()
]
ax.legend(handles=legenda, frameon=False, ncol=4, loc="upper right", fontsize=8)

plt.tight_layout()
fig.savefig(IMAGES_DIR / "casos_por_mes.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sintomas = [
    "fever",
    "myalgia",
    "headache",
    "rash",
    "vomiting",
    "nausea",
    "back_pain",
    "conjunctivitis",
    "arthritis",
    "joint_pain",
    "petechiae",
    "retro_orbital_pain",
    "tourniquet_test",
]

nomes_sintomas = {
    "fever": "Febre",
    "myalgia": "Mialgia",
    "headache": "Cefaleia",
    "rash": "Exantema",
    "vomiting": "Vômito",
    "nausea": "Náusea",
    "back_pain": "Dor nas costas",
    "conjunctivitis": "Conjuntivite",
    "arthritis": "Artrite",
    "joint_pain": "Dor articular",
    "petechiae": "Petéquias",
    "retro_orbital_pain": "Dor retro-orbital",
    "tourniquet_test": "Prova do laço",
}

def frequencia_sintomas(base):
    percentuais = {}
    for col in sintomas:
        serie = pd.to_numeric(base[col], errors="coerce")
        valido = serie.isin([1, 2]).sum()
        percentuais[nomes_sintomas[col]] = 0 if valido == 0 else (serie.eq(1).sum() / valido) * 100
    return pd.Series(percentuais)

freq_confirmados = frequencia_sintomas(df[df["final_classification"] == 1])
freq_descartados = frequencia_sintomas(df[df["final_classification"] == 0])
freq_geral = frequencia_sintomas(df)

sintomas_top = freq_geral.sort_values(ascending=False).head(10).index
comparativo = pd.DataFrame({
    "Confirmados": freq_confirmados.reindex(sintomas_top),
    "Descartados": freq_descartados.reindex(sintomas_top),
}).sort_values("Confirmados")

y = np.arange(len(comparativo))
altura = 0.36

fig, ax = plt.subplots(figsize=(11, 6.5))
bars_confirmados = ax.barh(
    y + altura / 2,
    comparativo["Confirmados"],
    height=altura,
    color="#0f766e",
    alpha=0.9,
    label="Confirmados",
)
bars_descartados = ax.barh(
    y - altura / 2,
    comparativo["Descartados"],
    height=altura,
    color="#64748b",
    alpha=0.85,
    label="Descartados",
)

ax.set_title("Sintomas Mais Frequentes por Classificação", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Percentual de casos válidos (%)", fontsize=9)
ax.set_ylabel("")
ax.set_yticks(y)
ax.set_yticklabels(comparativo.index)
ax.grid(axis="x", linestyle="--", alpha=0.22)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0, labelsize=8)
ax.tick_params(axis="y", length=0, labelsize=9)

for bars in [bars_confirmados, bars_descartados]:
    for bar in bars:
        largura = bar.get_width()
        ax.text(
            largura + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"{largura:.1f}%",
            va="center",
            ha="left",
            fontsize=8,
            color="#334155",
        )

ax.set_xlim(0, comparativo.max().max() * 1.18)
ax.legend(frameon=False, loc="lower right", fontsize=9)
plt.tight_layout()
fig.savefig(IMAGES_DIR / "sintomas_confirmados_vs_descartados.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

ordem_escolaridade = [
    "Analfabeto",
    "1ª a 4ª série incompleta",
    "4ª série completa",
    "5ª à 8ª série incompleta",
    "Ensino fundamental completo",
    "Ensino médio incompleto",
    "Ensino médio completo",
    "Educação superior incompleta",
    "Educação superior completa",
    "Não se aplica",
    "Ignorado",
]

casos = (
    df["education_level_label"]
    .value_counts()
    .reindex(ordem_escolaridade)
    .fillna(0)
)

plt.figure(figsize=(10, 6))
casos.plot(kind="barh")
plt.title("Casos de dengue por escolaridade")
plt.xlabel("Número de casos")
plt.ylabel("Escolaridade")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "casos_por_escolaridade.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

bins = [0, 9, 19, 29, 39, 49, 59, 69, 120]
labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]

df["faixa_etaria"] = pd.cut(df["age"], bins=bins, labels=labels, right=True)

casos = (
    df["faixa_etaria"]
    .value_counts()
    .reindex(labels)
)

plt.figure(figsize=(10, 6))
casos.plot(kind="bar")
plt.title("Casos de dengue por faixa etária")
plt.xlabel("Faixa etária")
plt.ylabel("Número de casos")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "casos_por_faixa_etaria.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

casos = (
    df["residence_state_label"]
    .value_counts()
    .sort_values()
)

plt.figure(figsize=(10, 8))
casos.plot(kind="barh")
plt.title("Casos de dengue por estado de residência")
plt.xlabel("Número de casos")
plt.ylabel("Estado")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "casos_por_estado_residencia.png", dpi=200, bbox_inches="tight")
plt.show()